In [25]:
# imports
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
# load data
movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

In [27]:
movies.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [28]:
ratings.head(6)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400


In [29]:
# Preprocessing (Content-Based)
movies['genres'] = movies['genres'].str.replace('|', ' ')
movies['tags'] = movies['genres']

In [30]:
movies.head()

,movieId,title,genres,tags
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy


In [31]:
# Vectorization
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(movies['tags']).toarray()

In [32]:
print(type(vectors))
print(vectors.shape)

<class 'numpy.ndarray'>
(9742, 23)


In [33]:
# Similarity (Content-Based)
similarity = cosine_similarity(vectors)
print(similarity.shape)
# First 5x5 rows and columns dekho
print(similarity[:5, :5])

(9742, 9742)
[[1.         0.77459667 0.31622777 0.25819889 0.4472136 ]
 [0.77459667 1.         0.         0.         0.        ]
 [0.31622777 0.         1.         0.81649658 0.70710678]
 [0.25819889 0.         0.81649658 1.         0.57735027]
 [0.4472136  0.         0.70710678 0.57735027 1.        ]]


In [34]:
# Content Function
def content_recommend(movie):
    movie_index = movies[movies['title'] == movie].index[0]
    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:10]

    return [movies.iloc[i[0]].title for i in movies_list]

In [7]:
# Collaborative Setup
movie_ratings = ratings.merge(movies, on='movieId')

pt = movie_ratings.pivot_table(index='title', columns='userId', values='rating')
pt.fillna(0, inplace=True)

similarity_scores = cosine_similarity(pt)

In [8]:
# Collaborative Function
def collaborative_recommend(movie):
    if movie not in pt.index:
        return []

    index = pt.index.get_loc(movie)
    distances = similarity_scores[index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:10]

    return [pt.index[i[0]] for i in movies_list]

In [9]:
# Hybrid Function
def hybrid_recommend(movie):
    content_movies = content_recommend(movie)
    collab_movies = collaborative_recommend(movie)

    final = list(dict.fromkeys(content_movies + collab_movies))

    return final[:10]

In [10]:
# test
hybrid_recommend("Toy Story (1995)")

['Antz (1998)',
 'Toy Story 2 (1999)',
 'Adventures of Rocky and Bullwinkle, The (2000)',
 "Emperor's New Groove, The (2000)",
 'Monsters, Inc. (2001)',
 'Wild, The (2006)',
 'Shrek the Third (2007)',
 'Tale of Despereaux, The (2008)',
 'Asterix and the Vikings (Astérix et les Vikings) (2006)',
 'Jurassic Park (1993)']